# 대칭행렬과 양의 정부호

> 선형대수 15강 · 대칭행렬과 양의 정부호

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [대칭행렬과 양의 정부호](https://mioon1402.github.io/timeseriesdata/linalg/L15-symmetric.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 왜 대칭행렬이 자주 나오나

**15-0. AᵀA 는 항상 대칭이다**

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

rng = np.random.default_rng(0)
for m, n in [(5, 3), (3, 7), (4, 4)]:
    A = rng.normal(size=(m, n))
    S = A.T @ A
    print(f"A 가 {m}×{n} → AᵀA 는 {S.shape}, 대칭인가: {np.allclose(S, S.T)}")

print()
print("증명: (AᵀA)ᵀ = Aᵀ(Aᵀ)ᵀ = AᵀA   ← 4강의 (AB)ᵀ = BᵀAᵀ")

## 1. 스펙트럼 정리 — S = QΛQᵀ

## 2. 양의 정부호 — 네 가지 동치 조건

## 3. 이차 형식과 그릇 모양

## 4. 타원의 축이 곧 고유벡터

## 5. Rayleigh 몫과 PCA

## 6. numpy 로 확인하기

**15-1. 스펙트럼 정리 확인**

In [ ]:
S = np.array([[4., 1.],
              [1., 3.]])

λ, Q = np.linalg.eigh(S)          # 대칭 전용 (13강)
print("고윳값 =", λ, "  ← 전부 실수")
print("\n고유벡터 Q =")
print(Q)
print("\nQᵀQ =")
print(np.round(Q.T @ Q, 12), "  ← 직교행렬")
print("\nQ Λ Qᵀ =")
print(Q @ np.diag(λ) @ Q.T)
print("S 와 같은가:", np.allclose(Q @ np.diag(λ) @ Q.T, S))

**15-2. 외적 합으로도 같다**

In [ ]:
합 = sum(λ[i] * np.outer(Q[:, i], Q[:, i]) for i in range(2))
print("Σ λᵢ qᵢqᵢᵀ =")
print(합)
print("S 와 같은가:", np.allclose(합, S))
print()
for i in range(2):
    P = np.outer(Q[:, i], Q[:, i])
    print(f"q{i+1}q{i+1}ᵀ 가 투영행렬인가:  P²=P {np.allclose(P@P, P)},  Pᵀ=P {np.allclose(P.T, P)}")

**15-3. 양의 정부호 — 네 조건이 전부 일치하는가**

In [ ]:
def 판정(S, 이름):
    λ = np.linalg.eigvalsh(S)

    # 피봇 (소거)
    U = S.astype(float).copy()
    피봇 = []
    ok = True
    for i in range(len(U)):
        if abs(U[i, i]) < 1e-12:
            ok = False
            break
        피봇.append(U[i, i])
        for j in range(i + 1, len(U)):
            U[j] -= U[j, i] / U[i, i] * U[i]

    # 선행 주소행렬식
    소 = [np.linalg.det(S[:k, :k]) for k in range(1, len(S) + 1)]

    # 촐레스키
    try:
        np.linalg.cholesky(S)
        chol = True
    except np.linalg.LinAlgError:
        chol = False

    print(f"[{이름}]")
    print(f"  ② 고윳값     {np.round(λ, 4)}         전부 > 0 : {bool((λ > 1e-12).all())}")
    print(f"  ③ 피봇       {np.round(피봇, 4) if ok else '소거 실패'}  전부 > 0 : {ok and all(p > 1e-12 for p in 피봇)}")
    print(f"  ④ 소행렬식   {np.round(소, 4)}         전부 > 0 : {all(d > 1e-12 for d in 소)}")
    print(f"  촐레스키 성공: {chol}")
    print()

판정(np.array([[4., 1.], [1., 3.]]), "양의 정부호")
판정(np.array([[1., 1.], [1., 1.]]), "반정부호 (λ 하나가 0)")
판정(np.array([[1., 2.], [2., 1.]]), "부정부호 (말안장)")

**15-4. 에너지 xᵀSx 를 직접 재보기**

In [ ]:
rng = np.random.default_rng(0)
x들 = rng.normal(size=(2000, 2))

for 이름, Mx in [("PD",  np.array([[4., 1.], [1., 3.]])),
                 ("부정", np.array([[1., 2.], [2., 1.]]))]:
    에너지 = np.einsum('ij,jk,ik->i', x들, Mx, x들)      # 각 x 에 대해 xᵀSx
    print(f"[{이름}]  최소 {에너지.min():8.3f}   최대 {에너지.max():8.3f}   "
          f"음수 비율 {(에너지 < 0).mean():.1%}")

print()
print("→ PD 는 어떤 x 를 넣어도 양수. 부정부호는 방향에 따라 부호가 바뀐다.")

**15-5. Rayleigh 몫의 최대·최소**

In [ ]:
S = np.array([[4., 1.], [1., 3.]])
λ = np.linalg.eigvalsh(S)

각도 = np.linspace(0, np.pi, 721)
값 = [(np.array([np.cos(t), np.sin(t)]) @ S @ np.array([np.cos(t), np.sin(t)]))
      for t in 각도]

print(f"Rayleigh 몫의 최솟값 = {min(값):.6f}")
print(f"           최댓값 = {max(값):.6f}")
print(f"고윳값             = {λ}")
print()
best = 각도[int(np.argmax(값))]
print(f"최댓값이 나오는 방향 = {np.round([np.cos(best), np.sin(best)], 4)}")
print(f"가장 큰 고윳값의 고유벡터 = {np.round(np.linalg.eigh(S)[1][:, -1], 4)}")
print("→ 같은 방향 (부호는 다를 수 있음)")

**15-6. PCA 맛보기**

In [ ]:
# 길쭉한 방향으로 퍼진 데이터를 만든다
rng = np.random.default_rng(1)
기본 = rng.normal(size=(400, 2)) * np.array([3.0, 0.6])
각 = np.deg2rad(30)
회전 = np.array([[np.cos(각), -np.sin(각)], [np.sin(각), np.cos(각)]])
X = 기본 @ 회전.T

X = X - X.mean(axis=0)                 # 평균을 0으로 (PCA 의 필수 단계)
C = np.cov(X, rowvar=False)            # 공분산 행렬 — 대칭이다

print("공분산 행렬 =")
print(C, "  대칭:", np.allclose(C, C.T))
print()

λc, Qc = np.linalg.eigh(C)
순서 = np.argsort(λc)[::-1]            # 큰 것부터
λc, Qc = λc[순서], Qc[:, 순서]

print("고윳값(=각 주성분의 분산) =", np.round(λc, 4))
print("설명 비율 =", np.round(λc / λc.sum(), 4))
print()
print("제1주성분 방향 =", np.round(Qc[:, 0], 4))
print("실제 만든 방향 =", np.round([np.cos(각), np.sin(각)], 4), " ← 30°")
print()
print("→ 데이터가 가장 넓게 퍼진 방향을 정확히 찾아냈다.")

**15-7. 연습문제**

In [ ]:
# 문제 1. [[2,0],[0,5]] 는 양의 정부호인가요? 타원의 축 방향과 길이는?

# 문제 2. [[1,3],[3,1]] 의 고윳값을 구하고 어떤 모양인지 판정하세요.

# 문제 3. 임의의 행렬 A 에 대해 AᵀA 는 항상 PSD 입니다.
#         왜 그럴까요? (힌트: xᵀAᵀAx = ‖Ax‖²)
#         언제 PD 가 될까요?

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
# 문제 1 — 대각행렬이므로 고윳값이 그대로, 축은 x·y축
A1 = np.array([[2., 0.], [0., 5.]])
λ1 = np.linalg.eigvalsh(A1)
print("문제 1: λ =", λ1, " 전부 양수 → PD")
print("        축 반지름 = 1/√λ =", np.round(1/np.sqrt(λ1), 4))
print("        λ 가 큰 y축 방향이 더 짧다")

# 문제 2 — 4 와 -2 → 부정부호(말안장)
A2 = np.array([[1., 3.], [3., 1.]])
print("\n문제 2: λ =", np.linalg.eigvalsh(A2), " → 부호가 섞임 = 말안장")

# 문제 3
rng2 = np.random.default_rng(3)
for m, n in [(5, 3), (2, 4)]:
    A3 = rng2.normal(size=(m, n))
    λ3 = np.linalg.eigvalsh(A3.T @ A3)
    print(f"\n문제 3: A 가 {m}×{n}, AᵀA 의 λ = {np.round(λ3, 6)}")
    print(f"        최소 고윳값 {λ3.min():.2e}, 랭크 {np.linalg.matrix_rank(A3)}/{n}")
print("\nxᵀAᵀAx = (Ax)ᵀ(Ax) = ‖Ax‖² ≥ 0 이므로 항상 PSD.")
print("Ax = 0 인 x≠0 가 없으면(= 열이 독립이면) 항상 > 0 이라 PD 가 된다.")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)